In [1]:
import pandas as pd
import requests
import json
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon 



In [2]:
demographics = json.loads(requests.get('https://files.jcrayb.com/files/ie300/majority.json').text)

In [3]:
centroids = {}

chicago = gpd.read_file('data/zips.geojson').dropna(subset='zip')[['zip', 'geometry']]
tracts = gpd.read_file('osmnx/data/tracts.geojson')



tract_demographics = {}

zips = chicago.zip.to_list()

zips = [zip for zip in zips if zip]

demographics = {zip: majority for zip, majority in demographics.items() if zip in zips}

for tract in tracts.iloc:
    centroid = tract.geometry.centroid

    for row in chicago.iloc:
        zip_geometry = row.geometry

        if zip_geometry.contains(centroid) and (row.zip in demographics):
            tract_demographics[tract.namelsad10] = demographics[row.zip]
            continue


In [4]:

json.dump(tract_demographics, open('data/tract_demographics.json', 'w'))

In [19]:
query = 'CHI'

destination = json.load(open(f'computation_results/budget_paths/{query}-0-1.json', 'r'))

print(len(destination))


res = {}

res_fastest = {}

res_by_race = {

}

import pandas

df = pd.DataFrame(index=list(tract_demographics.keys()), columns=['race', 'distance', 'restricted_distance', 'rdiff', 'adiff', 'n_cameras'])

#print(df)
for tract, race in tract_demographics.items():
    n_cameras = []
    unrestricted_time_to_reach = []
    restricted_time_to_reach = []
    if not tract in destination: continue
    dest = destination[tract]
    budgets = list(dest.keys())
    highest, lowest = (budgets[0], budgets[-1])
    #print(highest, lowest)
    #if highest == lowest: continue

    n_cameras += [int(highest)]
    unrestricted_time_to_reach += [dest[highest]]
    restricted_time_to_reach += [dest[lowest]]

    mean_uttr = np.mean(unrestricted_time_to_reach)
    mean_rttr = np.mean(restricted_time_to_reach)

    numbers = [race, mean_uttr, mean_rttr, (mean_rttr-mean_uttr)/mean_uttr*100, mean_rttr-mean_uttr, n_cameras[0]]
    #print(df[tract])
    df.loc[df.index == tract] = numbers

    res[tract] = {
        'demographic majority': race,
        'Avg time to reach destination': mean_uttr,
        'Avg time to reach destination while avoiding cameras': mean_rttr,
        '% difference in time': (mean_rttr-mean_uttr)/mean_uttr*100,
        'absolute difference in time': mean_rttr-mean_uttr,
        'Avg n cameras': np.mean(n_cameras)
    }

    
    closest_destination  = np.argmin(unrestricted_time_to_reach)

    min_uttr = unrestricted_time_to_reach[closest_destination]
    min_rttr = restricted_time_to_reach[closest_destination]

    if not n_cameras[closest_destination]: continue

print(len(res))

res_by_race = {

}

for tract, results in res.items():
    metrics = list(results.keys())
    dem_maj = results['demographic majority']

    if not dem_maj in res_by_race:
        res_by_race[dem_maj] = {metric: [] for metric in metrics if metric != 'demographic majority'}

    for metric in metrics:
        if metric == 'demographic majority':
            continue
        res_by_race[dem_maj][metric] += [results[metric]]
total_results = {}



for race, res in res_by_race.items():
    total_results[race] = {}
    for metric, values in res.items():
        total_results[race][metric] = (np.mean(values), np.std(values))

    total_results[race]['n'] = len(values)

json.dump(total_results, open(f'./analysis/{query}.json', 'w'), indent = 2)

801
784


In [20]:
df.to_csv(f'./analysis/{query}.csv')

In [22]:
query = 'MDW'

res = json.load(open(f'./analysis/{query}.json', 'r'))

n1 = res['black']['n']
n2 = res['white']['n']

res['T0'] = {}

for metric in res['black']:
    if metric == 'n': continue

    bmean, bstdev = res['black'][metric]
    wmean, wstdev = res['white'][metric]

    T0 = (bmean-wmean)/np.sqrt(bstdev**2/n1+wstdev**2/n2)
    res['T0'][metric] = T0
    print(metric, round(T0, 4))

json.dump(res, open(f'./analysis/{query}.json', 'w'), indent=2)

Avg time to reach destination -1.1101
Avg time to reach destination while avoiding cameras -0.3928
% difference in time 5.6644
absolute difference in time 5.823
Avg n cameras 8.6925


In [6]:
total_results

{'black': {'Avg time to reach destination': (824.3639670591363,
   238.05710260102234),
  'Avg time to reach destination while avoiding cameras': (852.8182276620697,
   245.8235002707183),
  '% difference in time': (3.54895450972817, 2.660591938528006),
  'absolute difference in time': (28.45426060293337, 22.34188132663986),
  'Avg n cameras': (1.2389157832191888, 0.8263601358890397)},
 'white': {'Avg time to reach destination': (743.9999773094596,
   246.33702311912103),
  'Avg time to reach destination while avoiding cameras': (787.6389687723249,
   262.7710650509443),
  '% difference in time': (5.935561093896997, 3.865760133977106),
  'absolute difference in time': (43.63899146286524, 30.529239596778055),
  'Avg n cameras': (2.052083502024882, 1.0829674119890411)}}